In [201]:
import torch
import torch.nn.functional as F
from typing import Any
import math

In [241]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', pretrained=True)
model

Using cache found in /Users/misha/.cache/torch/hub/pytorch_vision_v0.10.0
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /Users/misha/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this

In [242]:

class AblatedModule(torch.nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.reshape(x, self.output_shape(x))

    def reshape(self, x: torch.Tensor, shape) -> torch.Tensor:
        x = x.flatten()
        in_size = x.shape[0]
        out_size = math.prod(shape)
        if in_size > out_size:
            return x[:out_size].reshape(shape)
        else:
            return F.pad(x, (0, out_size - in_size, )).reshape(shape)

    def output_shape(self, x):
        raise NotImplementedError()

def cast_to_tuple_2d(value: int | tuple[int, int]) -> tuple[int, int]:
    if isinstance(value, int):
        return (value, value)
    else:
        return value

class AblatedAbstract2d(AblatedModule):
    kernel_size: tuple[int, int]
    stride: tuple[int, int]
    padding: tuple[int, int]
    dilation: tuple[int, int]

    def __init__(self, layer: torch.nn.Module) -> None:
        super().__init__()
        self.kernel_size = cast_to_tuple_2d(layer.kernel_size)
        self.stride = cast_to_tuple_2d(layer.stride)
        self.padding = cast_to_tuple_2d(layer.padding)
        self.dilation = cast_to_tuple_2d(layer.dilation)

    def output_shape(self, x) -> tuple[int, int, int, int]:
        x_shape = x.shape
        n = x_shape[0]
        c = x_shape[1]
        round_ = math.floor
        h_out = round_((x_shape[2] + 2 * self.padding[0] - self.dilation[0] * (self.kernel_size[0] - 1) - 1) / self.stride[0] + 1) 
        w_out = round_((x_shape[3] + 2 * self.padding[1] - self.dilation[1] * (self.kernel_size[1] - 1) - 1) / self.stride[1] + 1)
        return (n, c, h_out, w_out)

    def extra_repr(self) -> str:
        return (
            f"kernel_size={self.kernel_size}, stride={self.stride}, padding={self.padding}"
            f", dilation={self.dilation}"
        )

class AblatedConv2d(AblatedAbstract2d):
    out_channels: tuple[int]
    
    def __init__(self, layer: torch.nn.Conv2d) -> None:
        super().__init__(layer)
        self.out_channels = layer.out_channels

    def output_shape(self, x) -> tuple[int, int, int, int]:
        shape = super().output_shape(x)
        return (
            shape[0],
            self.out_channels,
            shape[2],
            shape[3],
        )

    def extra_repr(self) -> str:
        return f"out_channels={self.out_channels}, {super().extra_repr()}"

Pool2d = torch.nn.MaxPool2d

class AblatedPool2d(AblatedAbstract2d):
    pass

class AblatedAdaptivePool2d(AblatedModule):
    def __init__(self, layer: torch.nn.AdaptiveAvgPool2d) -> None:
        super().__init__()
        self.output_size = layer.output_size

    def output_shape(self, x):
        shape = x.shape
        return (shape[0], shape[1], self.output_size[0], self.output_size[1])

    def extra_repr(self) -> str:
        return f"output_size={self.output_size}"

class AblatedLinear(AblatedModule):
    out_features: int
    
    def __init__(self, layer: torch.nn.Linear) -> None:
        super().__init__()
        self.out_features = layer.out_features

    def output_shape(self, x):
        shape = x.shape
        return (shape[0], self.out_features)

    def extra_repr(self) -> str:
        return f"out_features={self.out_features}"

layer_types_for_ablation = [
    "Linear",
    "MaxPool2d",
    "Conv2d",
    "AdaptiveAvgPool2d",
]

def get_layers_for_ablation(model: torch.nn.Module) -> list[list[str]]:
    if len(model._modules) == 0:
        return

    results = []
    
    for key, module in model._modules.items():
        if len(module._modules) == 0:
            module_name = module._get_name()
            if module_name in layer_types_for_ablation:
                results.append([key])
        else:
            detected_submodules = get_layers_for_ablation(module)
            for submodule in detected_submodules:
                results.append([key] + submodule)
    return results

def ablate(layer: torch.nn.Module) -> torch.nn.Module:
    layer_name = layer._get_name()
    if layer_name in ["Conv2d"]:
        return AblatedConv2d(layer)
    elif layer_name in ["MaxPool2d"]:
        return AblatedPool2d(layer)
    elif layer_name in ["Linear"]:
        return AblatedLinear(layer)
    elif layer_name in ["AdaptiveAvgPool2d"]:
        return AblatedAdaptivePool2d(layer)

def ablate_by_key(model: torch.nn.Module, key: list[str]) -> torch.nn.Module:
    if len(key) == 1:
        model._modules[key[0]] = ablate(model._modules[key[0]])
    else:
        model._modules[key[0]] = ablate_by_key(model._modules[key[0]], key[1:])

    return model

In [243]:
keys = get_layers_for_ablation(model)
new_model = model
for key in keys:
    new_model = ablate_by_key(new_model, key)


In [244]:
new_model

AlexNet(
  (features): Sequential(
    (0): AblatedConv2d(out_channels=64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2), dilation=(1, 1))
    (1): ReLU(inplace=True)
    (2): AblatedPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0), dilation=(1, 1))
    (3): AblatedConv2d(out_channels=192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), dilation=(1, 1))
    (4): ReLU(inplace=True)
    (5): AblatedPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0), dilation=(1, 1))
    (6): AblatedConv2d(out_channels=384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), dilation=(1, 1))
    (7): ReLU(inplace=True)
    (8): AblatedConv2d(out_channels=256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), dilation=(1, 1))
    (9): ReLU(inplace=True)
    (10): AblatedConv2d(out_channels=256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), dilation=(1, 1))
    (11): ReLU(inplace=True)
    (12): AblatedPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0), dilation=(1, 1))
  )

In [246]:
x = torch.ones(1, 3, 224, 224)
model(x)

tensor([[0., 0., 0., 0., 0., 4., 0., 0., 4., 0., 4., 0., 0., 0., 0., 0., 0., 4.,
         4., 0., 4., 0., 0., 0., 0., 4., 0., 0., 0., 0., 0., 4., 0., 4., 0., 0.,
         0., 0., 4., 0., 0., 4., 0., 0., 0., 4., 0., 4., 4., 4., 0., 0., 0., 0.,
         4., 0., 0., 4., 0., 0., 0., 0., 0., 0., 0., 0., 4., 0., 0., 0., 0., 0.,
         4., 4., 0., 0., 0., 0., 4., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 4.,
         4., 4., 0., 4., 0., 0., 0., 0., 0., 4., 0., 0., 0., 4., 0., 0., 0., 0.,
         0., 0., 4., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 4., 0., 0.,
         0., 0., 0., 0., 4., 4., 0., 0., 4., 0., 4., 4., 0., 4., 4., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 4., 0., 0., 0., 0., 0., 0., 0., 4., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 4., 4., 0., 0., 0., 0., 0., 0., 4., 0.,
         4., 0., 0., 0., 0., 4., 0., 0., 0., 0., 4., 0., 0., 0., 0., 4., 0., 0.,
         0., 0., 0., 0., 4., 0., 0., 4., 0., 0., 0., 0., 4., 0., 4., 0., 4., 0.,
         0., 0., 4., 0., 0.,